# Week 05 - Dask Programming Assignment
Date : 02-23-2025 <br>
Author : Daipayan Bera

In [142]:
#Importing libraries
import dask.dataframe as dd
import pandas as pd
import numpy as np

1. Obtain the dataset from here: https://coronadatascraper.com/timeseries.csv.zip. Details and some interesting work associated with it is found here: https://coronadatascraper.com/#home
2. Extract the .csv file from the zipped archive, and load it into a Dask dataframe.

In [143]:
covid_dd = dd.read_csv("https://coronadatascraper.com/timeseries.csv", dtype={'name':str,'level':str,'city':object,'county':str,'state':str,'country':str,'population':np.float64,'lat':float,'long':float,'url':str,'aggregate':object,'tz':str,'cases':np.float64,'deaths':np.float64,'recovered':np.float64,'active':np.float64,'tested':np.float64,'hospitalized':np.float64,'hospitalized_current':np.float64,'discharged':np.float64,'icu':np.float64,'icu_current':np.float64,'growthFactor':np.float64,'date':object})

Please note: This dataset only consists of a total of 7 months in 2020 (January-July) and not 14 months as described in the statement. While American Samoa, Guam, Northern Mariana Islands, Puerto Rico, and Virgin Islands don't exactly come under the US, as it is stated in the dataset, therefore I didn't omit it.

3.1. Create a new dataframe object that consists of samples (i.e., rows) corresponding to states in the US.

In [144]:
us_df = covid_dd.query("country == 'United States'")
us_df = us_df.persist() #Using persist after the query as we are going to use this dask dataframe throughout this workfile

3.2. During the time period 2020-Jan-01 to 2021-Feb-28, rank states in terms of their per-capita mortality? Compute per-capita mortality during a specific period as the ratio of total deaths during that time period, to the average population of the state (compute the average population during the time period).

In [146]:
#The following code is going to use the dask dataframe us_df, therefore it is going to be executed parallelly
agg_df = us_df.groupby("state").agg({
    "population" : "mean",
    "deaths" : "sum"
})

state_df = agg_df.compute() #As we are saving the result as a pandas dataframe therefore, we don't need to use persist
state_df["Per-capita-death"] = state_df["deaths"]/state_df["population"]

In [147]:
state_df.sort_values("Per-capita-death", ascending=False)[["Per-capita-death"]]

,Per-capita-death
state,
New York,6.395701
Michigan,3.204753
Louisiana,2.735288
Illinois,2.043863
New Jersey,2.031200
Georgia,2.026085
Pennsylvania,1.831572
Virginia,1.359063
Mississippi,1.356705


3.3. During the same time period, compute the case fatality rate (CFR) per month, using one of the approaches defined in this scientific brief from the World Health Organization: https://www.who.int/news-room/commentaries/detail/estimating-mortality-from-covid-19Links to an external site.. This computation should yield an array of dimensions 50 (states) X 14 (months). State the assumptions you are making in computing this metric.

In [148]:
#The following code is going to use the dask dataframe us_df, therefore it is going to be executed parallelly
us_df["date"] = dd.to_datetime(us_df["date"])
us_df["month_year"] = us_df["date"].dt.strftime("%B %Y")

agg_us_df = us_df.groupby(["state","month_year"]).agg({
    "deaths": "sum",
    "cases": "sum"  
})

CFR = agg_us_df.compute() #Here, we can use compute directly without using persist as we won't be reusing this again

In [149]:
CFR["cfr_val"] = (CFR["deaths"]*100/CFR["cases"]).fillna(0)

In [150]:
pivot_CFR = CFR.pivot_table(index="state", columns="month_year", values="cfr_val").fillna(0)

In [151]:
#Printing out the result
pivot_CFR

month_year,April 2020,February 2020,January 2020,July 2020,June 2020,March 2020,May 2020
state,,,,,,,
Alabama,2.830899,0.000000,0.0,2.381771,2.962907,0.532313,3.889270
Alaska,2.314519,0.000000,0.0,1.207417,1.303247,0.335008,2.196905
American Samoa,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Arizona,1.486545,0.000000,0.0,0.973523,0.211513,0.000000,1.992175
Arkansas,1.911450,0.000000,0.0,1.274360,1.515155,0.915656,2.129628
California,3.479974,0.000000,0.0,2.543001,3.178666,2.006735,3.983350
Colorado,2.636616,0.000000,0.0,5.063691,5.419220,0.939250,5.372019
Connecticut,6.477626,0.000000,0.0,9.500807,9.383106,1.814771,9.016204
Delaware,2.734038,0.000000,0.0,4.013148,4.164733,1.334107,3.574849


3.4. Using this matrix as input, compute the ranking of states, on how the CFR rate changed over time. This computation will involve an aggregation of of month-to-month changes in CFR. Note that some of these month-to-month changes can be positive (CFR increases from a previous month to the current month) or negative (CFR decrease from a previous month to the current month), or zero (no change in CFR). You need to aggregate these individual changes, across all time periods.

In [152]:
#Reordering the month columns
pivot_CFR = pivot_CFR[["January 2020", "February 2020", "March 2020", "April 2020", "May 2020", "June 2020", "July 2020"]]

In [153]:
#Using diff function on each column, then summarizing with sum function and applying sorting in descending order
pivot_CFR.diff(axis=1).sum(axis=1).sort_values(ascending=False)

state
Connecticut                     9.500807
Michigan                        8.623360
New Jersey                      8.251406
Massachusetts                   7.437951
Pennsylvania                    7.418973
New York                        7.233961
Northern Mariana Islands        6.535948
New Hampshire                   6.436738
United States Virgin Islands    5.976096
Rhode Island                    5.821287
Washington, D.C.                5.330110
Indiana                         5.314359
Ohio                            5.275945
Louisiana                       5.138853
Colorado                        5.063691
Illinois                        4.802791
Maryland                        4.649307
Vermont                         4.559147
Delaware                        4.013148
New Mexico                      4.003795
Missouri                        3.979086
Minnesota                       3.958075
Washington                      3.921734
Mississippi                     3.743453
Kentucky  

4. For each of the above operations, explain using appropriate reasoning, whether using a parallelized and/or distributed way of performing the computation makes sense.

In this exercise, I used parallelized tasks with Dask for certain computations, while relying on sequential execution for the rest, as it was more efficient in those cases. As per the computation graph that we saw in the lecture video (dask2-3.mp4 - By Dr. Mudigonda) helped illustrate when parallelized or distributed computing should be leveraged—specifically when the dataset is too large to fit into local memory or when the computation is highly CPU-intensive.

In this case, even though the dataset could easily fit into local memory, the computations could still have been time-consuming if we had to query the dataset repeatedly. To address this, I utilized parallelized operations through Dask during data querying and aggregation, as these tasks tend to be the most computation-heavy.

Additionally, I used Dask's persist() method after querying the Dask DataFrame to ensure that the query results are not recomputed multiple times when needed. This approach optimized performance by storing intermediate results in memory. If the dataset had been significantly larger, a distributed cluster could have been used to scale the computations further.